In [356]:
import pandas as pd

In [357]:
mql = pd.read_csv("olist_marketing_qualified_leads_dataset.csv")
closed = pd.read_csv("olist_closed_deals_dataset.csv")
orders = pd.read_csv("olist_orders_dataset.csv")
order_items = pd.read_csv("olist_order_items_dataset.csv")
sellers = pd.read_csv("olist_sellers_dataset.csv")
customers = pd.read_csv("olist_customers_dataset.csv")
reviews = pd.read_csv("olist_order_reviews_dataset.csv")

In [358]:
print("MQL exact duplicates:", mql.duplicated().sum())
print("Closed exact duplicates:", closed.duplicated().sum())
print("Orders exact duplicates:", orders.duplicated().sum())
print("Order Items exact duplicates:", order_items.duplicated().sum())
print("Sellers exact duplicates:", sellers.duplicated().sum())
print("Customers exact duplicates:", customers.duplicated().sum())
print("Reviews exact duplicates:", reviews.duplicated().sum())

MQL exact duplicates: 0
Closed exact duplicates: 0
Orders exact duplicates: 0
Order Items exact duplicates: 0
Sellers exact duplicates: 0
Customers exact duplicates: 0
Reviews exact duplicates: 0


In [359]:
print("Duplicate order_id + order_item_id:",order_items[["order_id", "order_item_id"]].duplicated().sum())


Duplicate order_id + order_item_id: 0


In [360]:
mql["first_contact_date"] = pd.to_datetime(mql["first_contact_date"])
closed["won_date"] = pd.to_datetime(closed["won_date"])
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"])
orders["order_estimated_delivery_date"] = pd.to_datetime(orders["order_estimated_delivery_date"])

In [361]:
orders["order_status"] == "delivered"

0        True
1        True
2        True
3        True
4        True
         ... 
99436    True
99437    True
99438    True
99439    True
99440    True
Name: order_status, Length: 99441, dtype: bool

In [362]:
review_by_order = (reviews.groupby("order_id")["review_score"].mean().reset_index())

In [363]:
review_by_order

,order_id,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,5.0
1,00018f77f2f0320c557190d7a144bdd3,4.0
2,000229ec398224ef6ca0657da4fc703e,5.0
3,00024acbcdf0a6daa1e931b038114c75,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0
...,...,...
98668,fffc94f6ce00a00581880bf54a75a037,5.0
98669,fffcd46ef2263f404302a634eb57f7eb,5.0
98670,fffce4705a9662cd70adb13d4a31832d,5.0
98671,fffe18544ffabc95dfada21779c9644f,5.0


In [364]:
print(review_by_order.shape)
print("Duplicate order IDs:", review_by_order["order_id"].duplicated().sum())

(98673, 2)
Duplicate order IDs: 0


In [365]:
delivered_orders = orders[orders["order_status"] == "delivered"].copy()

In [366]:
delivered_orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26
...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15


In [367]:
#removing rows without date of delivery 
delivered_orders = delivered_orders.dropna(subset=["order_delivered_customer_date"]).copy()

In [368]:
delivered_orders["delivery_days"] = (delivered_orders["order_delivered_customer_date"]- delivered_orders["order_purchase_timestamp"]).dt.days

In [369]:
delivered_orders["delivery_days"].describe()

count    96470.000000
mean        12.093604
std          9.551380
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
Name: delivery_days, dtype: float64

In [370]:
#build and validate seller orders

In [371]:
seller_order_level_all = (order_items.groupby(["seller_id", "order_id"]).agg(order_value=("price", "sum"),freight_value=("freight_value", "sum"),item_count=("order_item_id", "count")).reset_index())


In [372]:
print("Seller-order rows:",seller_order_level_all.shape)

print("Duplicate seller-order combinations:",seller_order_level_all[["seller_id", "order_id"]].duplicated().sum())

Seller-order rows: (100010, 5)
Duplicate seller-order combinations: 0


In [373]:
seller_order_level_all = seller_order_level_all.merge(orders[["order_id","customer_id","order_status","order_purchase_timestamp","order_delivered_customer_date","order_estimated_delivery_date"]],on="order_id",how="left")

In [374]:
print("Seller-order rows after orders merge:",len(seller_order_level_all))
print("Missing order status:",seller_order_level_all["order_status"].isna().sum())

Seller-order rows after orders merge: 100010
Missing order status: 0


In [375]:
seller_order_level_all = seller_order_level_all.merge(review_by_order,on="order_id",how="left")

In [376]:
print("Rows after review merge:",len(seller_order_level_all))

print("Missing review scores:",seller_order_level_all["review_score"].isna().sum())

Rows after review merge: 100010
Missing review scores: 763


In [377]:
seller_order_level_all = seller_order_level_all.merge(sellers,on="seller_id",how="left")

In [378]:
print("Rows after seller merge:",len(seller_order_level_all))

print("Missing seller state:",seller_order_level_all["seller_state"].isna().sum())

print("Missing seller city:",seller_order_level_all["seller_city"].isna().sum())

Rows after seller merge: 100010
Missing seller state: 0
Missing seller city: 0


In [379]:
seller_order_level_delivered = seller_order_level_all[seller_order_level_all["order_status"] == "delivered"].copy()

In [380]:
print("Delivered seller-order rows:",len(seller_order_level_delivered))

print("Unique delivered sellers:",seller_order_level_delivered["seller_id"].nunique())

Delivered seller-order rows: 97819
Unique delivered sellers: 2970


In [381]:
seller_summary_delivered = (seller_order_level_delivered.groupby("seller_id").agg(total_orders=("order_id", "nunique"),total_revenue=("order_value", "sum"),average_review_score=("review_score", "mean"),review_count=("review_score", "count")).reset_index())

In [382]:
print("Seller summary shape:",seller_summary_delivered.shape)

print("Duplicate seller IDs:",seller_summary_delivered["seller_id"].duplicated().sum())

Seller summary shape: (2970, 5)
Duplicate seller IDs: 0


In [383]:
seller_summary_delivered = seller_summary_delivered.merge(sellers,on="seller_id",how="left")

In [384]:
print("Seller summary rows:",len(seller_summary_delivered))

print("Duplicate seller IDs:",seller_summary_delivered["seller_id"].duplicated().sum())

print("Missing seller state:",seller_summary_delivered["seller_state"].isna().sum())

print("Missing seller city:",seller_summary_delivered["seller_city"].isna().sum())

Seller summary rows: 2970
Duplicate seller IDs: 0
Missing seller state: 0
Missing seller city: 0


In [385]:
print("Minimum orders:",seller_summary_delivered["total_orders"].min())
print("Maximum orders:",seller_summary_delivered["total_orders"].max())
print("Minimum sales value:",seller_summary_delivered["total_revenue"].min())
print("Maximum sales value:",seller_summary_delivered["total_revenue"].max())
print("Minimum review score:",seller_summary_delivered["average_review_score"].min())
print("Maximum review score:",seller_summary_delivered["average_review_score"].max())

Minimum orders: 1
Maximum orders: 1819
Minimum sales value: 6.5
Maximum sales value: 226987.93
Minimum review score: 1.0
Maximum review score: 5.0


In [386]:
funnel_clean = mql.merge(closed,on="mql_id",how="left")

In [387]:
print("MQL rows:", len(mql))
print("Funnel rows:", len(funnel_clean))

MQL rows: 8000
Funnel rows: 8000


In [388]:
converted_sellers = funnel_clean[funnel_clean["seller_id"].notna()].copy()

In [389]:
print("Converted sellers:",len(converted_sellers))
print("Duplicate seller IDs:",converted_sellers["seller_id"].duplicated().sum())

Converted sellers: 842
Duplicate seller IDs: 0


In [390]:
seller_analysis = converted_sellers.merge(seller_summary_delivered,on="seller_id",how="left")

In [391]:
print("Converted sellers:",len(converted_sellers))
print("Seller analysis rows:",len(seller_analysis))
print("Duplicate seller IDs:",seller_analysis["seller_id"].duplicated().sum())

Converted sellers: 842
Seller analysis rows: 842
Duplicate seller IDs: 0


In [392]:
print("Sellers with delivered performance:",seller_analysis["total_orders"].notna().sum())
print("Sellers without delivered performance:",seller_analysis["total_orders"].isna().sum())

Sellers with delivered performance: 376
Sellers without delivered performance: 466


In [393]:
print(seller_analysis[["total_orders","total_revenue","average_review_score","seller_state"]].isna().sum())

total_orders            466
total_revenue           466
average_review_score    466
seller_state            466
dtype: int64


In [394]:
print("First MQL contact:",mql["first_contact_date"].min())
print("Last MQL contact:",mql["first_contact_date"].max())
print("First won seller:",closed["won_date"].min())
print("Last won seller:",closed["won_date"].max())
print("First order:",orders["order_purchase_timestamp"].min())
print("Last order:",orders["order_purchase_timestamp"].max())

First MQL contact: 2017-06-14 00:00:00
Last MQL contact: 2018-05-31 00:00:00
First won seller: 2017-12-05 02:00:00
Last won seller: 2018-11-14 18:04:19
First order: 2016-09-04 21:15:19
Last order: 2018-10-17 17:30:18


In [395]:
print("Missing order status after merge:",seller_order_level_all["order_status"].isna().sum())
print("Missing seller state after merge:",seller_order_level_all["seller_state"].isna().sum())
print("Missing review score after merge:",seller_order_level_all["review_score"].isna().sum())

Missing order status after merge: 0
Missing seller state after merge: 0
Missing review score after merge: 763
